In [4]:
import numpy as np
from pathlib import Path


def load_dataset(path="/home/gmarihuan/processed/chbmit_windows_all.npz"):
    """Return the whole dataset as X (float32), y, patient."""
    d = np.load(path, allow_pickle=True)
    X = d["X"].astype(np.float32)
    y = d["y"]
    patient = d["patient_id"]
    return X, y, patient


X, y, patient = load_dataset()
print(X.shape)

(7208, 21, 768)


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SqueezeExcitation(nn.Module):
    """Squeeze-and-Excitation block (Hu et al.), Eq. (6)-(8) in the paper."""
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        reduced = max(channels // reduction, 1)
        # No bias -> matches the 128-parameter count reported in Table 2
        self.fc1 = nn.Linear(channels, reduced, bias=False)
        self.fc2 = nn.Linear(reduced, channels, bias=False)

    def forward(self, x):
        b, c, h, w = x.shape
        # Squeeze: global average pooling (Eq. 6)
        z = x.mean(dim=(2, 3))                      # (B, C)
        # Excitation: two FC layers with ReLU + Sigmoid gating (Eq. 7)
        s = F.relu(self.fc1(z))
        s = torch.sigmoid(self.fc2(s))               # (B, C)
        s = s.view(b, c, 1, 1)
        # Scale: channel-wise recalibration (Eq. 8)
        return x * s


class DCSENet(nn.Module):
    """
    Dilated Convolutional Squeeze-and-Excitation Network (DCSENets)
    for patient-independent EEG spectrogram seizure classification.

    Input: (B, 3, 224, 224) spectrogram images
    Output: (B, 1) seizure probability (sigmoid)
    """
    def __init__(self, in_channels: int = 3, dilation: int = 2,
                 se_reduction: int = 16, dropout_p: float = 0.5):
        super().__init__()

        # --- Block 1: Dilated Conv -> MaxPool -> SE ---
        # k=3, dilation=2 => output shrinks by (k-1)*(r-1)+ (k-1) = 4 px per dim
        # 224 -> 220 (matches Table 2)
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, dilation=dilation)
        self.pool1 = nn.MaxPool2d(kernel_size=2)          # 220 -> 110
        self.se1 = SqueezeExcitation(32, reduction=se_reduction)

        # --- Block 2: Dilated Conv -> MaxPool -> SE ---
        # 110 -> 106
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, dilation=dilation)
        self.pool2 = nn.MaxPool2d(kernel_size=2)          # 106 -> 53
        self.se2 = SqueezeExcitation(32, reduction=se_reduction)

        self.dropout = nn.Dropout(dropout_p)

        # Flatten: 53 * 53 * 32 = 89,888 (matches Table 2)
        flat_dim = 53 * 53 * 32
        self.fc1 = nn.Linear(flat_dim, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        # Block 1
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = self.se1(x)

        # Block 2
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.se2(x)

        x = self.dropout(x)
        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))  # binary: seizure vs. normal
        return x


if __name__ == "__main__":
    model = DCSENet()
    dummy = torch.randn(2, 3, 224, 224)
    out = model(dummy)
    print("Output shape:", out.shape)  # torch.Size([2, 1])

    n_params = sum(p.numel() for p in model.parameters())
    print(f"Total trainable parameters: {n_params:,}")  # ~5,763,361 as in Table 2

Output shape: torch.Size([2, 1])
Total trainable parameters: 5,763,361


In [3]:
import numpy as np
import torch
from scipy.interpolate import interp1d
from PIL import Image


def hann_taper(N: int) -> np.ndarray:
    """Hann taper function, Eq. (1)."""
    n = np.arange(N)
    return 0.5 * (1 - np.cos(2 * np.pi * n / N))


def gaussian_taper(N: int, sigma: float) -> np.ndarray:
    """Gaussian taper function, Eq. (2)."""
    n = np.arange(N)
    return np.exp(-0.5 * ((n - N / 2) / (sigma * N / 2)) ** 2)


def get_taper(window_func: str, N: int, sigma: float = 0.1) -> np.ndarray:
    if window_func == "hann":
        return hann_taper(N)
    elif window_func == "gaussian":
        return gaussian_taper(N, sigma)
    elif window_func == "none":
        return np.ones(N)
    else:
        raise ValueError(f"Unknown window_func: {window_func}")


def eeg_window_to_spectrogram(
    eeg_window: np.ndarray,     # shape (Ch, T) raw EEG samples for the window
    fs: int = 256,              # sampling frequency (Hz), Section 2.1
    window_length_s: float = 1.0,   # WL: 1s, 5s or 10s per the ablation study
    step_s: float = None,       # hop between STFT time points; defaults to window_length_s
    f_min: float = 0.5,
    f_max: float = 40.0,
    n_freq_bins: int = 224,     # log-spaced frequency bins (F)
    window_func: str = "hann",  # "hann", "gaussian", or "none"
    sigma: float = 0.1,         # only used for gaussian taper (tune per WL, Section 3.2.2)
) -> np.ndarray:
    """
    Implements Algorithm 1 (Enhanced STFT Spectrogram Visualization) for a
    single EEG window, producing a per-channel time-frequency spectrogram
    stack of shape (Ch, n_freq_bins, n_time_bins).

    eeg_window: (Ch, T) array of raw voltage samples (e.g. one channel-set
                segment extracted around a seizure/normal event).
    Returns: AllCh, shape (Ch, F, T_idx) in dB.
    """
    n_ch, n_samples = eeg_window.shape

    # TWS: time-window samples (line 4 of Algorithm 1)
    TWS = int(round(window_length_s * fs))
    step = int(round((step_s or window_length_s) * fs))

    # Time indices within the time range (T_idx), line 3
    T_idx = np.arange(TWS // 2, n_samples - TWS // 2 + 1, step)
    if len(T_idx) == 0:
        T_idx = np.array([n_samples // 2])

    # Log-spaced frequency bins between f_min and f_max Hz, line 2
    freqs_log = np.geomspace(f_min, f_max, n_freq_bins)

    # Taper (window) function, line 5
    WF = get_taper(window_func, TWS, sigma=sigma)

    AllCh = np.zeros((n_ch, n_freq_bins, len(T_idx)), dtype=np.float64)

    fft_freqs = np.fft.rfftfreq(TWS, d=1.0 / fs)  # frequency axis of the FFT

    for ch in range(n_ch):
        STFT = np.zeros((n_freq_bins, len(T_idx)), dtype=np.float64)

        for j, t_center in enumerate(T_idx):
            start_idx = max(t_center - TWS // 2, 0)
            end_idx = min(t_center + TWS // 2, n_samples)

            data_seg = eeg_window[ch, start_idx:end_idx]

            # Zero-pad if segment is shorter than TWS (edge windows)
            if len(data_seg) < TWS:
                pad = TWS - len(data_seg)
                data_seg = np.pad(data_seg, (0, pad), mode="constant")

            # Apply taper (line 13)
            data_tapered = data_seg * WF

            # FFT and power spectrum (lines 14-15)
            y = np.fft.rfft(data_tapered, n=TWS)
            P = np.abs(y) ** 2

            # Interpolate power spectrum onto log-spaced frequencies (line 16)
            interp_fn = interp1d(
                fft_freqs, P, kind="linear",
                bounds_error=False, fill_value=(P[0], P[-1])
            )
            STFT[:, j] = interp_fn(freqs_log)

        # Convert to dB relative to the channel's mean power (line 18)
        mean_power = np.mean(STFT) + 1e-12
        AllCh[ch] = 10 * np.log10(STFT / mean_power + 1e-12)

    return AllCh  # (Ch, F, T_idx), in dB


def spectrogram_to_model_input(
    spectrogram_2d: np.ndarray,   # single-channel spectrogram, shape (F, T)
    image_size: int = 224,
) -> torch.Tensor:
    """
    Converts one channel's dB spectrogram (e.g. F3-C3 as in Fig. 3e/f) into
    the resized 3-channel image tensor expected by DCSENet's input layer.

    Paper: "Initial generated images through Algorithm 1 are of size
    462 x 775 but resized to 224 x 224" (Section 2.3).
    """
    # Normalize to [0, 255] for image-like representation
    spec = spectrogram_2d.copy()
    spec -= spec.min()
    denom = spec.max() + 1e-12
    spec = (spec / denom * 255).astype(np.uint8)

    # Resize to 224x224 using PIL (bilinear), then replicate to 3 channels
    img = Image.fromarray(spec).resize((image_size, image_size), Image.BILINEAR)
    img_arr = np.array(img, dtype=np.float32) / 255.0        # (224, 224)
    img_rgb = np.stack([img_arr] * 3, axis=0)                # (3, 224, 224)

    return torch.from_numpy(img_rgb).float()


def eeg_window_to_dcsenet_input(
    eeg_window: np.ndarray,
    channel_idx: int = 0,
    fs: int = 256,
    window_length_s: float = 1.0,
    window_func: str = "hann",
    sigma: float = 0.1,
) -> torch.Tensor:
    """
    End-to-end: raw multichannel EEG window -> single (3, 224, 224) tensor
    ready to feed into DCSENet.forward(). Selects one channel's spectrogram
    (e.g. the channel of interest for the epileptogenic zone) as in Fig. 3(e,f).
    """
    all_ch_spectrograms = eeg_window_to_spectrogram(
        eeg_window, fs=fs, window_length_s=window_length_s,
        window_func=window_func, sigma=sigma,
    )
    channel_spectrogram = all_ch_spectrograms[channel_idx]     # (F, T)
    model_input = spectrogram_to_model_input(channel_spectrogram)
    return model_input.unsqueeze(0)  # add batch dim -> (1, 3, 224, 224)


if __name__ == "__main__":
    # Simulate a 22-channel, 4-second EEG window sampled at 256 Hz
    fs = 256
    n_channels = 22
    duration_s = 4
    eeg_window = np.random.randn(n_channels, fs * duration_s) * 20  # µV-scale noise

    x = eeg_window_to_dcsenet_input(
        eeg_window, channel_idx=5, fs=fs,
        window_length_s=1.0, window_func="hann",
    )
    print("Model input shape:", x.shape)  # torch.Size([1, 3, 224, 224])

    model = DCSENet()  # from the previous artifact
    with torch.no_grad():
        prob = model(x)
    print("Seizure probability:", prob.item())

Model input shape: torch.Size([1, 3, 224, 224])
Seizure probability: 0.5194100737571716


In [6]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.interpolate import interp1d
from PIL import Image
from pathlib import Path


# ---------------------------------------------------------------------------
# Taper functions (Eq. 1, 2) and single-channel STFT spectrogram (Algorithm 1)
# ---------------------------------------------------------------------------

def hann_taper(N: int) -> np.ndarray:
    n = np.arange(N)
    return 0.5 * (1 - np.cos(2 * np.pi * n / N))


def gaussian_taper(N: int, sigma: float) -> np.ndarray:
    n = np.arange(N)
    return np.exp(-0.5 * ((n - N / 2) / (sigma * N / 2)) ** 2)


def get_taper(window_func: str, N: int, sigma: float = 0.1) -> np.ndarray:
    if window_func == "hann":
        return hann_taper(N)
    elif window_func == "gaussian":
        return gaussian_taper(N, sigma)
    elif window_func == "none":
        return np.ones(N)
    raise ValueError(f"Unknown window_func: {window_func}")


def channel_to_spectrogram(
    signal_1d: np.ndarray,      # (T,) raw samples for ONE channel
    fs: int = 256,
    window_length_s: float = 1.0,
    step_s: float = None,
    f_min: float = 0.5,
    f_max: float = 40.0,
    n_freq_bins: int = 224,
    window_func: str = "hann",
    sigma: float = 0.1,
) -> np.ndarray:
    """Algorithm 1 applied to a single-channel 1D EEG segment -> (F, T_idx) dB."""
    n_samples = signal_1d.shape[0]
    TWS = int(round(window_length_s * fs))
    step = int(round((step_s or window_length_s) * fs))

    T_idx = np.arange(TWS // 2, max(n_samples - TWS // 2, TWS // 2) + 1, step)
    if len(T_idx) == 0:
        T_idx = np.array([n_samples // 2])

    freqs_log = np.geomspace(f_min, f_max, n_freq_bins)
    WF = get_taper(window_func, TWS, sigma=sigma)
    fft_freqs = np.fft.rfftfreq(TWS, d=1.0 / fs)

    STFT = np.zeros((n_freq_bins, len(T_idx)), dtype=np.float64)

    for j, t_center in enumerate(T_idx):
        start_idx = max(t_center - TWS // 2, 0)
        end_idx = min(t_center + TWS // 2, n_samples)
        seg = signal_1d[start_idx:end_idx]

        if len(seg) < TWS:
            seg = np.pad(seg, (0, TWS - len(seg)), mode="constant")

        tapered = seg * WF
        y = np.fft.rfft(tapered, n=TWS)
        P = np.abs(y) ** 2

        interp_fn = interp1d(
            fft_freqs, P, kind="linear", bounds_error=False,
            fill_value=(P[0], P[-1]),
        )
        STFT[:, j] = interp_fn(freqs_log)

    mean_power = np.mean(STFT) + 1e-12
    return 10 * np.log10(STFT / mean_power + 1e-12)   # (F, T_idx)


def spectrogram_to_image_tensor(spec_2d: np.ndarray, image_size: int = 224) -> torch.Tensor:
    """Normalize + resize a (F,T) dB spectrogram to a (3,224,224) tensor."""
    spec = spec_2d.copy()
    spec -= spec.min()
    spec /= (spec.max() + 1e-12)
    spec = (spec * 255).astype(np.uint8)

    img = Image.fromarray(spec).resize((image_size, image_size), Image.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0          # (224, 224)
    arr = np.stack([arr, arr, arr], axis=0)                  # (3, 224, 224)
    return torch.from_numpy(arr)


# ---------------------------------------------------------------------------
# Dataset loader (as given) + lazy spectrogram Dataset
# ---------------------------------------------------------------------------

def load_dataset(path="/home/gmarihuan/processed/chbmit_windows_all.npz"):
    """Return the whole dataset as X (float32), y, patient."""
    d = np.load(path, allow_pickle=True)
    X = d["X"].astype(np.float32)
    y = d["y"]
    patient = d["patient_id"]
    return X, y, patient


class CHBMITSpectrogramDataset(Dataset):
    """
    Wraps raw EEG windows (N, Ch, T) and lazily converts them to DCSENet-ready
    spectrogram tensors (3, 224, 224) on __getitem__, following Algorithm 1.

    channel_mode:
      - "single":  always use `channel_idx` (e.g. the F3-C3 channel from Fig. 3)
      - "random":  pick a random channel per __getitem__ (data augmentation
                    across channels, useful since the paper trains per-channel)
      - "fixed_list": cycle through a supplied list of channel indices,
                    expanding the effective dataset size by len(channel_idx)
    """

    def __init__(
        self,
        X: np.ndarray,
        y: np.ndarray,
        patient: np.ndarray,
        indices: np.ndarray = None,
        fs: int = 256,
        window_length_s: float = 1.0,
        window_func: str = "hann",
        sigma: float = 0.1,
        image_size: int = 224,
        channel_mode: str = "single",
        channel_idx: int = 5,
        channel_list: list = None,
    ):
        self.X = X
        self.y = y
        self.patient = patient
        self.indices = indices if indices is not None else np.arange(len(X))

        self.fs = fs
        self.window_length_s = window_length_s
        self.window_func = window_func
        self.sigma = sigma
        self.image_size = image_size

        self.channel_mode = channel_mode
        self.channel_idx = channel_idx
        self.channel_list = channel_list or list(range(X.shape[1]))

        if channel_mode == "fixed_list":
            # Expand: every (sample, channel) pair becomes one dataset item
            self._expanded = [
                (i, ch) for i in self.indices for ch in self.channel_list
            ]
        else:
            self._expanded = None

    def __len__(self):
        if self._expanded is not None:
            return len(self._expanded)
        return len(self.indices)

    def _get_sample_and_channel(self, idx):
        if self._expanded is not None:
            sample_idx, ch = self._expanded[idx]
        else:
            sample_idx = self.indices[idx]
            if self.channel_mode == "random":
                ch = np.random.choice(self.channel_list)
            else:  # "single"
                ch = self.channel_idx
        return sample_idx, ch

    def __getitem__(self, idx):
        sample_idx, ch = self._get_sample_and_channel(idx)

        raw_signal = self.X[sample_idx, ch, :]           # (T,)
        spec = channel_to_spectrogram(
            raw_signal,
            fs=self.fs,
            window_length_s=self.window_length_s,
            window_func=self.window_func,
            sigma=self.sigma,
        )
        img_tensor = spectrogram_to_image_tensor(spec, image_size=self.image_size)

        label = torch.tensor(float(self.y[sample_idx]), dtype=torch.float32)
        pid = self.patient[sample_idx]

        return img_tensor, label, pid


# ---------------------------------------------------------------------------
# Leave-one-patient-out split (Section 2.5 / Fig. 4)
# ---------------------------------------------------------------------------

def lopocv_split(patient: np.ndarray, test_patient_id) -> tuple[np.ndarray, np.ndarray]:
    """
    Returns (train_indices, test_indices) for one LOPOCV fold, leaving the
    given patient entirely out of training, as in Fig. 4.
    """
    patient = np.asarray(patient)
    test_mask = patient == test_patient_id
    train_idx = np.where(~test_mask)[0]
    test_idx = np.where(test_mask)[0]
    return train_idx, test_idx


def build_lopocv_loaders(
    X, y, patient,
    test_patient_id,
    batch_size: int = 32,
    window_length_s: float = 1.0,
    window_func: str = "hann",
    sigma: float = 0.1,
    channel_idx: int = 5,
    num_workers: int = 4,
):
    """Convenience wrapper: build train/test DataLoaders for one LOPOCV fold."""
    train_idx, test_idx = lopocv_split(patient, test_patient_id)

    train_ds = CHBMITSpectrogramDataset(
        X, y, patient, indices=train_idx,
        window_length_s=window_length_s, window_func=window_func, sigma=sigma,
        channel_mode="single", channel_idx=channel_idx,
    )
    test_ds = CHBMITSpectrogramDataset(
        X, y, patient, indices=test_idx,
        window_length_s=window_length_s, window_func=window_func, sigma=sigma,
        channel_mode="single", channel_idx=channel_idx,
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=num_workers, drop_last=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, drop_last=False)
    return train_loader, test_loader


# ---------------------------------------------------------------------------
# Example usage
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    X, y, patient = load_dataset()
    print("Raw dataset:", X.shape, y.shape, patient.shape)

    unique_patients = np.unique(patient)
    print("Patients:", unique_patients)

    # One LOPOCV fold: leave the first patient out for testing
    held_out = unique_patients[0]
    train_loader, test_loader = build_lopocv_loaders(
        X, y, patient,
        test_patient_id=held_out,
        batch_size=16,
        window_length_s=1.0,
        window_func="hann",
        channel_idx=5,
    )

    imgs, labels, pids = next(iter(train_loader))
    print("Batch images:", imgs.shape)   # (16, 3, 224, 224)
    print("Batch labels:", labels.shape)
    print("Batch patients (should exclude held-out):", set(pids) - {held_out})

Raw dataset: (7208, 21, 768) (7208,) (7208,)
Patients: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]
Batch images: torch.Size([16, 3, 224, 224])
Batch labels: torch.Size([16])
Batch patients (should exclude held-out): {tensor(16, dtype=torch.int16), tensor(21, dtype=torch.int16), tensor(7, dtype=torch.int16), tensor(9, dtype=torch.int16), tensor(3, dtype=torch.int16), tensor(14, dtype=torch.int16), tensor(14, dtype=torch.int16), tensor(13, dtype=torch.int16), tensor(20, dtype=torch.int16), tensor(14, dtype=torch.int16), tensor(17, dtype=torch.int16), tensor(12, dtype=torch.int16), tensor(11, dtype=torch.int16), tensor(21, dtype=torch.int16), tensor(14, dtype=torch.int16), tensor(14, dtype=torch.int16)}


In [7]:
import copy
import numpy as np
import torch
import torch.nn as nn
from scipy.stats import ks_2samp
from sklearn.metrics import confusion_matrix


# ---------------------------------------------------------------------------
# Metrics (Eq. 9-11)
# ---------------------------------------------------------------------------

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Accuracy, sensitivity (recall/TPR), specificity (TNR)."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-12)
    sensitivity = tp / (tp + fn + 1e-12)
    specificity = tn / (tn + fp + 1e-12)
    return {
        "accuracy": accuracy * 100,
        "sensitivity": sensitivity * 100,
        "specificity": specificity * 100,
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
    }


# ---------------------------------------------------------------------------
# Single epoch train / eval passes
# ---------------------------------------------------------------------------

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    n_samples = 0

    for imgs, labels, _pids in loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).unsqueeze(1)  # (B, 1)

        optimizer.zero_grad()
        outputs = model(imgs)                 # (B, 1) sigmoid probs
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        n_samples += imgs.size(0)

    return running_loss / max(n_samples, 1)


@torch.no_grad()
def evaluate(model, loader, device, threshold: float = 0.5):
    model.eval()
    all_preds, all_labels = [], []

    for imgs, labels, _pids in loader:
        imgs = imgs.to(device, non_blocking=True)
        outputs = model(imgs).squeeze(1).cpu().numpy()   # (B,)
        preds = (outputs >= threshold).astype(int)

        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().astype(int).tolist())

    metrics = compute_metrics(np.array(all_labels), np.array(all_preds))
    return metrics


# ---------------------------------------------------------------------------
# Optional: KS test between train / test sample distributions (Section 3.4)
# ---------------------------------------------------------------------------

def ks_test_train_test(X: np.ndarray, train_idx: np.ndarray, test_idx: np.ndarray,
                        channel_idx: int = 5) -> float:
    """
    Kolmogorov-Smirnov test on the raw signal amplitude distribution of the
    chosen channel, comparing training vs test samples for one LOPOCV fold.
    Returns the p-value; p > 0.05 (or 0.01 < p <= 0.05) => distributions
    are not significantly different (paper's threshold, Section 3.4).
    """
    train_vals = X[train_idx, channel_idx, :].ravel()
    test_vals = X[test_idx, channel_idx, :].ravel()
    _, p_value = ks_2samp(train_vals, test_vals)
    return p_value


# ---------------------------------------------------------------------------
# Algorithm 2: iterative optimization for ONE LOPOCV fold
# ---------------------------------------------------------------------------

def train_one_fold(
    model_fn,                 # callable -> fresh nn.Module (e.g. DCSENet)
    train_loader,
    test_loader,
    device,
    epochs: int = 20,
    lr: float = 0.001,
    verbose: bool = True,
):
    """
    Implements Algorithm 2: trains for `epochs`, evaluates on the held-out
    patient after every epoch, and keeps the weights that achieved the best
    test accuracy so far.
    """
    model = model_fn().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    best_acc = -float("inf")
    best_state = None
    best_metrics = None
    history = []

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        metrics = evaluate(model, test_loader, device)
        history.append({"epoch": epoch, "train_loss": train_loss, **metrics})

        if verbose:
            print(
                f"  epoch {epoch:02d}/{epochs} | loss={train_loss:.4f} "
                f"| acc={metrics['accuracy']:.2f}% "
                f"| sens={metrics['sensitivity']:.2f}% "
                f"| spec={metrics['specificity']:.2f}%"
            )

        if metrics["accuracy"] > best_acc:
            best_acc = metrics["accuracy"]
            best_state = copy.deepcopy(model.state_dict())
            best_metrics = metrics

    model.load_state_dict(best_state)
    return model, best_metrics, history


# ---------------------------------------------------------------------------
# Fig. 4: full leave-one-patient-out cross-validation loop
# ---------------------------------------------------------------------------

def run_lopocv(
    X: np.ndarray,
    y: np.ndarray,
    patient: np.ndarray,
    model_fn,                       # e.g. lambda: DCSENet()
    device: torch.device = None,
    batch_size: int = 32,
    epochs: int = 20,
    lr: float = 0.001,
    window_length_s: float = 1.0,
    window_func: str = "hann",
    sigma: float = 0.1,
    channel_idx: int = 5,
    num_workers: int = 4,
    run_ks_test: bool = True,
    verbose: bool = True,
) -> dict:
    """
    Runs the full patient-independent LOPOCV evaluation (Fig. 4): for each
    patient P_i, trains on all other patients and tests exclusively on P_i,
    using Algorithm 2's iterative best-model selection per fold.

    Returns a dict with per-patient results and overall average accuracy /
    standard deviation, mirroring Table 3.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    unique_patients = np.unique(patient)
    fold_results = {}
    accuracies = []

    for fold_i, held_out in enumerate(unique_patients, start=1):
        if verbose:
            print(f"\n=== LOPOCV fold {fold_i}/{len(unique_patients)} "
                  f"| held-out patient: {held_out} ===")

        train_idx, test_idx = lopocv_split(patient, held_out)

        if run_ks_test:
            p_value = ks_test_train_test(X, train_idx, test_idx, channel_idx=channel_idx)
            if verbose:
                sig = "no significant difference" if p_value > 0.01 else "significant difference"
                print(f"  KS test p-value={p_value:.4f} ({sig} vs train distribution)")
        else:
            p_value = None

        train_loader, test_loader = build_lopocv_loaders(
            X, y, patient,
            test_patient_id=held_out,
            batch_size=batch_size,
            window_length_s=window_length_s,
            window_func=window_func,
            sigma=sigma,
            channel_idx=channel_idx,
            num_workers=num_workers,
        )

        _model, best_metrics, history = train_one_fold(
            model_fn, train_loader, test_loader, device,
            epochs=epochs, lr=lr, verbose=verbose,
        )

        fold_results[str(held_out)] = {
            "metrics": best_metrics,
            "ks_p_value": p_value,
            "history": history,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
        }
        accuracies.append(best_metrics["accuracy"])

        if verbose:
            print(f"  -> best fold accuracy: {best_metrics['accuracy']:.2f}%")

    summary = {
        "average_accuracy": float(np.mean(accuracies)),
        "std_accuracy": float(np.std(accuracies)),
        "per_patient": fold_results,
    }

    if verbose:
        print(
            f"\n=== LOPOCV complete: avg accuracy = "
            f"{summary['average_accuracy']:.2f} ± {summary['std_accuracy']:.2f}% ==="
        )

    return summary


# ---------------------------------------------------------------------------
# Example usage
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    from functools import partial

    X, y, patient = load_dataset()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    results = run_lopocv(
        X, y, patient,
        model_fn=lambda: DCSENet(),
        device=device,
        batch_size=32,
        epochs=20,
        lr=0.001,
        window_length_s=1.0,
        window_func="hann",
        channel_idx=5,
        num_workers=4,
    )

    print("\nPer-patient accuracy:")
    for pid, res in results["per_patient"].items():
        m = res["metrics"]
        print(f"  {pid}: acc={m['accuracy']:.2f}% sens={m['sensitivity']:.2f}% "
              f"spec={m['specificity']:.2f}% (KS p={res['ks_p_value']})")


=== LOPOCV fold 1/24 | held-out patient: 0 ===
  KS test p-value=0.0000 (significant difference vs train distribution)
  epoch 01/20 | loss=0.7066 | acc=50.00% | sens=0.00% | spec=100.00%
  epoch 02/20 | loss=0.6935 | acc=50.00% | sens=0.00% | spec=100.00%
  epoch 03/20 | loss=0.6932 | acc=51.03% | sens=2.05% | spec=100.00%
  epoch 04/20 | loss=0.6931 | acc=59.59% | sens=22.60% | spec=96.58%
  epoch 05/20 | loss=0.6911 | acc=48.97% | sens=3.42% | spec=94.52%
  epoch 06/20 | loss=0.6896 | acc=50.00% | sens=0.00% | spec=100.00%
  epoch 07/20 | loss=0.6932 | acc=50.00% | sens=0.00% | spec=100.00%
  epoch 08/20 | loss=0.6932 | acc=50.00% | sens=0.00% | spec=100.00%
  epoch 09/20 | loss=0.6932 | acc=50.00% | sens=0.00% | spec=100.00%
  epoch 10/20 | loss=0.6932 | acc=50.00% | sens=0.00% | spec=100.00%
  epoch 11/20 | loss=0.6932 | acc=50.00% | sens=0.00% | spec=100.00%
  epoch 12/20 | loss=0.6932 | acc=50.00% | sens=100.00% | spec=0.00%
  epoch 13/20 | loss=0.6932 | acc=50.00% | sens=0.00%